# Amaliy mashgʻulot 12. Mashina tarjimasi, soʻzlarni moslashtirish (Word Alignment) va Beam Search algoritmi

12-laboratoriyada EM ning har bir iteratsiyasida $t(f\mid e)$ tarjima jadvali qanday aniqlashishini va beam search kuchsiz gipotezalarni qanday tashlab yuborishini koʻrdingiz. Bu yerda buni kichik oʻzbekcha-ruscha parallel korpusda kodda takrorlaymiz (tarjima **oʻzbek tilidan rus tiliga**):

1. IBM Model 1 ni noldan yozamiz va EM $t(f\mid e)$ ehtimollarini iteratsiyadan iteratsiyaga qanday «oʻtkirlashini» kuzatamiz;
2. Viterbi moslashtirishlarini topamiz va `nltk.translate.IBMModel1` bilan solishtiramiz;
3. oʻzbek soʻzlarini morfemalarga ajratish moslashtirishga qanday yordam berishini koʻramiz («uyda» ↔ «в доме»);
4. oʻyinchoq «keyingi soʻz» modelida beam search ni yozamiz, uni ochkoʻz (greedy) qidiruv bilan solishtiramiz va uzunlik boʻyicha meʼyorlash nima uchun kerakligini tushunamiz.

*«NLP algoritmlari» kursining 12-laboratoriyasiga amaliy mashgʻulot. Kataklarni yuqoridan pastga ishga tushiring: Shift+Enter yoki «Runtime → Run all». Javoblaringizni saqlash uchun «File → Save a copy in Drive» ni bosing.*

## 1. IBM Model 1: EM noldan

IBM Model 1 oʻzbekcha $e = e_1\dots e_l$ gapdan ruscha $f = f_1\dots f_m$ gap qanday hosil boʻlishini tavsiflaydi: har bir ruscha $f_j$ soʻz bitta oʻzbekcha $e_{a_j}$ soʻzni (yoki boʻsh soʻz NULL ni) tanlaydi va $t(f_j \mid e_{a_j})$ ehtimol bilan hosil boʻladi. $a$ moslashtirishni koʻrmaymiz, shuning uchun $t$ ni EM algoritmi bilan oʻrganamiz:

- **E-qadam:** har bir $f_j$ uchun u har bir $e_i$ dan kelgan boʻlish ehtimolini hisoblaymiz: $P(a_j = i) = \dfrac{t(f_j\mid e_i)}{\sum_{i'} t(f_j\mid e_{i'})}$ va bu ulushlarni $c(f, e)$ hisoblagichlariga qoʻshamiz;
- **M-qadam:** $t(f\mid e) = c(f, e) / \sum_{f'} c(f', e)$.

Tekis jadvaldan boshlaymiz. Korpus laboratoriyadagi kabi juftlardan iborat: «uy» – «дом», «uyda» – «в доме», «kitobim» – «моя книга».

In [ ]:
corpus = [   # (oʻzbekcha, ruscha)
    ("uy", "дом"), ("katta uy", "большой дом"), ("yangi uy", "новый дом"),
    ("uyim", "мой дом"), ("uyda", "в доме"), ("uyimda", "в моём доме"),
    ("katta uyda", "в большом доме"), ("kitob", "книга"), ("katta kitob", "большая книга"),
    ("yangi kitob", "новая книга"), ("kitobim", "моя книга"), ("kitobda", "в книге"),
    ("maktab", "школа"), ("katta maktab", "большая школа"), ("maktabim", "моя школа"),
    ("maktabda", "в школе"), ("men kitob oʻqiyman", "я читаю книгу"), ("u kitob oʻqiydi", "он читает книгу"),
    ("men uyda yashayman", "я живу в доме"), ("u uyda yashaydi", "он живёт в доме"), ("u katta uyda yashaydi", "он живёт в большом доме"),
    ("men maktabda oʻqiyman", "я учусь в школе"),
]
corpus = [(s.split(), t.split()) for s, t in corpus]
print(len(corpus), "juft gap")

In [ ]:
import numpy as np
import pandas as pd

class IBM1:
    def __init__(self, corpus):
        self.corpus = corpus
        self.src = ["NULL"] + sorted({w for s, _ in corpus for w in s})     # e: oʻzbekcha soʻzlar + boʻsh soʻz NULL
        self.tgt = sorted({w for _, t in corpus for w in t})                # f: ruscha soʻzlar
        self.si = {w: i for i, w in enumerate(self.src)}
        self.ti = {w: j for j, w in enumerate(self.tgt)}
        # t[e, f] = t(f|e); boshlanishi: barcha tarjimalar teng ehtimolli
        self.t = np.full((len(self.src), len(self.tgt)), 1 / len(self.tgt))
        self.history, self.loglik = [self.t.copy()], []

    def em_step(self):
        count = np.zeros_like(self.t)
        ll = 0.0
        for s, f_sent in self.corpus:
            e_ids = [0] + [self.si[w] for w in s]              # 0-oʻrin = NULL
            for f in f_sent:
                j = self.ti[f]
                p = self.t[e_ids, j]
                ll += np.log(p.sum() / len(e_ids))
                np.add.at(count[:, j], e_ids, p / p.sum())     # E-qadam: P(a_j = i) ulushlari hisoblagichlarga qoʻshiladi
        self.t = count / count.sum(axis=1, keepdims=True)      # M-qadam: hisoblagichlarni har bir e boʻyicha meʼyorlaymiz
        self.history.append(self.t.copy())
        self.loglik.append(ll)

    def train(self, n_iter):
        for _ in range(n_iter):
            self.em_step()
        return self

    def prob(self, f, e, it=-1):
        return self.history[it][self.si[e], self.ti[f]]

    def align(self, s, f_sent):
        # har bir f uchun eng ehtimolli e ni olamiz (0 = NULL)
        e_words = ["NULL"] + s
        return [int(np.argmax([self.prob(f, e) for e in e_words])) for f in f_sent]

model = IBM1(corpus).train(20)
print(f"|V_e| = {len(model.src)} (NULL bilan), |V_f| = {len(model.tgt)}, boshida: t = 1/{len(model.tgt)} = {1 / len(model.tgt):.3f}")
print("iteratsiyalar boʻyicha log-likelihood:", " ".join(f"{x:.1f}" for x in model.loglik[:8]), "...", f"{model.loglik[-1]:.1f}")

watch = [("uy", "дом"), ("uyda", "доме"), ("uyda", "в"), ("NULL", "в"), ("katta", "большой"),
         ("katta", "большая"), ("men", "я"), ("oʻqiyman", "читаю")]
iters = [0, 1, 2, 3, 5, 10, 20]
table = pd.DataFrame({f"it {k}": [model.prob(f, e, k) for e, f in watch] for k in iters},
                     index=[f"t({f} | {e})" for e, f in watch])
print(table.round(3).to_string())

Korpusning haqqoniylik funksiyasi (log-likelihood) har bir iteratsiyada oshadi — bu EM kafolati. Jadvalda ehtimollar qanday «oʻtkirlashishi» koʻrinadi: boshida barcha tarjimalar teng ehtimolli ($1/|V_f| = 1/23 \approx 0.043$), bir necha iteratsiyadan keyin toʻgʻri tarjima katta massani oladi ($t(\text{я}\mid\text{men}) \approx 0.99$).

Uchta qiziq hol:
- ruscha «в» predlogi avval «uyda» ga, keyin esa boʻsh soʻz NULL ga «yopishadi»: $t(\text{в}\mid\text{NULL}) \to 0.99$;
- «katta» ning massasi ruscha «большая», «большой», «большом» shakllari orasida boʻlinadi: rus tilida sifat rod va kelishik boʻyicha oʻzgaradi. $t(\text{большой}\mid\text{katta})$ esa hatto kamayadi: «большой» korpusda faqat bir marta uchraydi va uni koʻproq «uy» «tortib oladi»;
- «oʻqiyman» ikki xil tarjima qilinadi («читаю» va «учусь»), shuning uchun $t(\text{читаю}\mid\text{oʻqiyman})$ 0.5 dan oshmaydi.

## 2. Viterbi moslashtirishlari va nltk bilan tekshirish

Har bir ruscha soʻz uchun eng yaxshi moslashtirish: $a_j = \arg\max_i t(f_j \mid e_i)$. IBM Model 1 ning muhim cheklovi: har bir ruscha soʻz **faqat bitta** oʻzbekcha soʻz (yoki NULL) bilan bogʻlanadi. Bu yoʻnalishda bitta oʻzbekcha soʻz bir nechta ruscha soʻzga bogʻlanishi mumkin, lekin «в» predlogi baribir NULL ga ketadi: model «uyda» ichidagi «-da» ni koʻrmaydi, chunki u uchun «uyda» va «uy» butunlay boshqa soʻzlar.

In [ ]:
def show_alignment(model, s, f_sent):
    a = model.align(s, f_sent)
    e_words = ["NULL"] + s
    links = "  ".join(f"{f}←{e_words[i]}" for f, i in zip(f_sent, a))
    lonely = [w for k, w in enumerate(s, start=1) if k not in a]
    print(f"{' '.join(s):<26} | {' '.join(f_sent):<26} {links}" + (f"   [juftsiz: {', '.join(lonely)}]" if lonely else ""))

for s, f_sent in corpus:
    show_alignment(model, s, f_sent)

Natijada ikkita xato bor: «katta uy» da «большой» «uy» ga, «u kitob oʻqiydi» da «книгу» «oʻqiydi» ga bogʻlandi. Ikkala ruscha shakl ham korpusda bor-yoʻgʻi bir marta uchraydi. «katta» ning ehtimol massasi beshta gapdagi turli ruscha shakllar orasida boʻlinib ketgan, «kitob» niki esa «книга» va «книгу» orasida, shuning uchun bir martalik shaklni qoʻshni soʻz «arzonroq» tushuntiradi. Bu IBM Model 1 ning kichik korpusdagi odatiy xatosi; korpus kattalashgani sari bunday xatolar kamayadi.

Chapda — oʻqitishdan keyingi $t(f\mid e)$ jadvalining bir qismi. Oʻngda — bitta gap uchun moslashtirishning aposterior ehtimollari $P(a_j = i)$, ramka bilan Viterbi moslashtirishi belgilangan.

In [ ]:
import matplotlib.pyplot as plt

s, f_sent = corpus[20]
e_words = ["NULL"] + s
P = np.array([[model.prob(f, e) for f in f_sent] for e in e_words])
P = P / P.sum(axis=0)                     # P(a_j = i): ustun boʻyicha meʼyorlaymiz

show_e = ["NULL", "uy", "uyda", "katta", "uyim", "kitob", "men", "yashayman", "u"]
show_f = ["в", "дом", "доме", "большой", "мой", "книга", "я", "живу", "он"]
T = np.array([[model.prob(f, e) for f in show_f] for e in show_e])

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5), gridspec_kw={"width_ratios": [1.3, 1]})
ax[0].imshow(T, cmap="Blues", vmin=0, vmax=1)
ax[0].set_xticks(range(len(show_f)))
ax[0].set_xticklabels(show_f, rotation=45, ha="right")
ax[0].set_yticks(range(len(show_e)))
ax[0].set_yticklabels(show_e)
ax[0].set_title("20 iteratsiyadan keyin t(f | e)")
for i in range(len(show_e)):
    for j in range(len(show_f)):
        if T[i, j] >= 0.05:
            ax[0].text(j, i, f"{T[i, j]:.2f}", ha="center", va="center", fontsize=8,
                       color="white" if T[i, j] > 0.6 else "black")
ax[1].imshow(P, cmap="Oranges", vmin=0, vmax=1)
for j, i in enumerate(model.align(s, f_sent)):
    ax[1].add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, ec="black", lw=2))
for i in range(len(e_words)):
    for j in range(len(f_sent)):
        ax[1].text(j, i, f"{P[i, j]:.2f}", ha="center", va="center", fontsize=8)
ax[1].set_xticks(range(len(f_sent)))
ax[1].set_xticklabels(f_sent)
ax[1].set_yticks(range(len(e_words)))
ax[1].set_yticklabels(e_words)
ax[1].set_title("P(a_j = i) va Viterbi moslashtirishi")
plt.tight_layout()
plt.show()

Xuddi shu model `nltk` da. Argumentlar tartibiga eʼtibor bering: `AlignedSent(words, mots)`, bu yerda `words` — hosil qilinadigan gap (bizda ruscha), `mots` — manba gap (oʻzbekcha). nltk da boʻsh soʻz `None` bilan belgilanadi.

In [ ]:
from nltk.translate import IBMModel1, AlignedSent

bitext = [AlignedSent(f_sent, s) for s, f_sent in corpus]    # nltk: words = hosil qilinadigan (rus.), mots = manba (oʻzb.)
ibm = IBMModel1(bitext, 20)

# birga uchraydigan soʻz juftlarini solishtiramiz (qolganlarini nltk yangilamaydi)
cooc = {(e, f) for s, f_sent in corpus for e in ["NULL"] + s for f in f_sent}
diff = max(abs(ibm.translation_table[f][None if e == "NULL" else e] - model.prob(f, e)) for e, f in cooc)
print(f"max |t_ours - t_nltk| = {diff:.2e}  ({len(cooc)} ta juft boʻyicha)")

same = 0
for (s, f_sent), b in zip(corpus, bitext):
    ours = {(j, i - 1 if i > 0 else None) for j, i in enumerate(model.align(s, f_sent))}
    same += ours == set(b.alignment)
print(f"mos kelgan moslashtirishlar: {same} / {len(corpus)}")
print(bitext[-2].words, bitext[-2].mots, "->", bitext[-2].alignment)

## 3. Morfemalar moslashtirishga yordam beradi

«uyda» = «uy» + «-da», va aynan «-da» ruscha «в» ga mos keladi. Oʻzbekcha soʻzlarni morfemalarga ajratsak, model «в» ni «+da» bilan, «мой/моя» ni «+im» bilan bogʻlay oladi. Quyida juda oddiy segmentator: agar kamida 2 harf qolsa, `-da` va `-im` ni kesib oladi. U faqat shu korpus uchun yozilgan; oʻzbek tili uchun tayyor morfologik tahlilchilar kam, haqiqiy tizimlarda odatda soʻz boʻlaklari (BPE, 2-mavzu) ishlatiladi.


Oʻqitishdan keyin bizda «oʻzbekcha morfema → ruscha soʻz» lugʻati boʻladi. Soʻzma-soʻz tarjima qilsa boʻladimi? Korpusda **yoʻq** iboralarda sinab koʻramiz. Ikki muammo koʻrinadi: soʻz tartibi (ruschada predlog soʻzdan oldin keladi) va soʻz shakllari (kelishik, rod). Haqiqiy tarjima tizimi butun gapning koʻp variantlarini solishtirib, tartib va shakllarni tanlashi kerak. Bu dekoderning ishi.

In [ ]:
def segment(word, suffixes=("da", "im")):
    # soʻz oxiridan -da va -im ni kesamiz: uyimda -> uy +im +da
    out = []
    peeled = True
    while peeled:
        peeled = False
        for suf in suffixes:
            if word.endswith(suf) and len(word) - len(suf) >= 2:
                out.insert(0, "+" + suf)
                word = word[: -len(suf)]
                peeled = True
    return [word] + out

def segment_sentence(words):
    return [m for w in words for m in segment(w)]

seg_corpus = [(segment_sentence(s), f_sent) for s, f_sent in corpus]
seg_model = IBM1(seg_corpus).train(20)
for k in [3, 4, 5, 6, 20]:
    show_alignment(seg_model, *seg_corpus[k])
print()
for e, f in [("+da", "в"), ("NULL", "в"), ("+im", "мой"), ("+im", "моя"), ("uy", "доме")]:
    print(f"t({f} | {e}) = {seg_model.prob(f, e):.3f}")

In [ ]:
def word_by_word(model, words):
    # har bir oʻzbekcha morfema uchun eng ehtimolli ruscha soʻzni olamiz
    return [model.tgt[int(np.argmax(model.t[model.si[w]]))] for w in words if w in model.si]

gold = {"katta maktabda": "в большой школе", "kitobimda": "в моей книге"}
for phrase in gold:
    words = segment_sentence(phrase.split())
    print(f"{phrase:<18} → {' '.join(word_by_word(seg_model, words)):<24} (toʻgʻrisi: {gold[phrase]})")

## 4. Beam search: dekoder

Neyron tarjimon tarjimani soʻzma-soʻz hosil qiladi: har qadamda model keyingi soʻz ustidan $P(y_t \mid y_{<t}, x)$ taqsimotini beradi. Butun tarjimaning ehtimoli — koʻpaytma: $\log P(y\mid x) = \sum_t \log P(y_t \mid y_{<t}, x)$.

- **Ochkoʻz (greedy) qidiruv** har qadamda eng ehtimolli soʻzni oladi — tez, lekin eng yaxshi tarjimani oʻtkazib yuborishi mumkin.
- **Beam search** $k$ ta eng yaxshi qisman gipotezani (nur) saqlaydi. Har qadamda ularning har birini barcha soʻzlar bilan davom ettiradi, saralaydi va $k$ ta eng yaxshisini qoldiradi. `</s>` bilan tugagan gipoteza chetga qoʻyiladi, nur esa 1 taga torayadi.

Quyida «Men yangi kitob oʻqiyman» gapini tarjima qilish uchun oʻyinchoq model: har bir prefiks uchun keyingi ruscha soʻzning taqsimoti berilgan (sonlar misol uchun oʻylab topilgan).

In [ ]:
EOS = "</s>"
# Men yangi kitob oʻqiyman  ->  P(keyingi soʻz | yozilgan prefiks)
lm = {
    (): {"я": .40, "читаю": .35, "новую": .25},
    ("я",): {"читаю": .90, "новую": .10},
    ("читаю",): {"книгу": .70, "новую": .30},
    ("новую",): {"книгу": .90, EOS: .10},
    ("я", "читаю"): {"книгу": .55, "новую": .45},
    ("я", "новую"): {"книгу": 1.0},
    ("читаю", "книгу"): {EOS: 1.0},
    ("читаю", "новую"): {"книгу": .95, EOS: .05},
    ("новую", "книгу"): {"читаю": .80, EOS: .20},
    ("я", "читаю", "книгу"): {EOS: 1.0},
    ("я", "читаю", "новую"): {"книгу": .90, EOS: .10},
    ("я", "новую", "книгу"): {"читаю": 1.0},
    ("читаю", "новую", "книгу"): {EOS: 1.0},
    ("новую", "книгу", "читаю"): {EOS: 1.0},
    ("я", "читаю", "новую", "книгу"): {EOS: 1.0},
    ("я", "новую", "книгу", "читаю"): {EOS: 1.0},
}

import math

def beam_search(lm, k, alpha=0.0, verbose=False):
    beams = [((), 0.0)]                 # (gipoteza, log P)
    finished = []
    step = 0
    while beams and len(finished) < k:
        step += 1
        cands = [(h + (w,), lp + math.log(p)) for h, lp in beams for w, p in lm[h].items()]
        cands.sort(key=lambda c: -c[1])
        room = k - len(finished)        # nurda nechta joy qoldi
        kept, pruned = cands[:room], cands[room:]
        beams = [c for c in kept if c[0][-1] != EOS]
        finished += [c for c in kept if c[0][-1] == EOS]
        if verbose:
            print(f"qadam {step}:")
            for h, lp in kept:
                tag = "(tugadi)" if h[-1] == EOS else ""
                print(f"   + {' '.join(h):<34} log P = {lp:6.2f} {tag}")
            for h, lp in pruned:
                print(f"   - {' '.join(h):<34} log P = {lp:6.2f}   tashlab yuborildi")
    score = lambda c: c[1] / len(c[0]) ** alpha    # uzunlik boʻyicha meʼyorlash; alpha = 0 — meʼyorlashsiz
    return max(finished, key=score), finished

best, _ = beam_search(lm, k=2, verbose=True)
print("\neng yaxshi tugagan gipoteza:", " ".join(best[0]), f"  P = {math.exp(best[1]):.3f}")

Ochkoʻz qidiruvni ($k=1$), turli kenglikdagi beam search ni va barcha gipotezalarni toʻliq koʻrib chiqadigan **aniq** qidiruvni solishtiramiz (daraxt kichik, shuning uchun toʻliq koʻrib chiqish mumkin; haqiqiy modelda gipotezalar soni astronomik).

**Uzunlik boʻyicha meʼyorlash.** Har bir soʻz ehtimolni < 1 songa koʻpaytiradi, shuning uchun qisqa gipotezalar deyarli har doim uzunlaridan ehtimolliroq: «читаю книгу» («я» va «новую» siz) $P$ boʻyicha yutadi. Meʼyorlash $\log P$ ni uzunlikning $\alpha$ darajasiga boʻladi: $\text{score} = \log P / L^{\alpha}$ ($L$ `</s>` ni ham hisoblaydi). $\alpha = 1$ da bu bitta soʻzga toʻgʻri keladigan oʻrtacha log-ehtimol.

In [ ]:
def all_hypotheses(lm, h=(), lp=0.0):
    # daraxtni toʻliq koʻrib chiqish: barcha tugagan gipotezalar
    if h and h[-1] == EOS:
        return [(h, lp)]
    return [x for w, p in lm[h].items() for x in all_hypotheses(lm, h + (w,), lp + math.log(p))]

hyps = all_hypotheses(lm)
print(f"jami gipotezalar: {len(hyps)}, ehtimollar yigʻindisi = {sum(math.exp(lp) for _, lp in hyps):.3f}")
for alpha in (0.0, 1.0):
    exact = max(hyps, key=lambda c: c[1] / len(c[0]) ** alpha)
    print(f"\nalpha = {alpha:g}   toʻliq koʻrib chiqishdagi aniq javob: {' '.join(exact[0])}")
    for k in range(1, 6):
        (h, lp), _ = beam_search(lm, k, alpha)
        print(f"   k = {k}: {' '.join(h):<32} P = {math.exp(lp):.3f}   log P / L^alpha = {lp / len(h) ** alpha:6.3f}")

Nima koʻrinadi: ochkoʻz qidiruv «новую» ni yoʻqotadi, chunki uchinchi qadamda «я читаю» dan keyin «книгу» (0.55) «новую» (0.45) dan biroz ehtimolliroq. Meʼyorlashsiz keng nur eng ehtimolli, lekin juda qisqa gipotezani topadi. Meʼyorlash bilan va $k \ge 4$ da toʻliq tarjima «я читаю новую книгу» topiladi. Grafikda chapda — gipoteza ehtimoli, oʻngda — bitta soʻzga oʻrtacha ehtimol $\exp(\log P / L)$; gʻolib toʻq sariq rangda.

In [ ]:
import matplotlib.pyplot as plt

hyps.sort(key=lambda c: c[1])
labels = [" ".join(h[:-1]) for h, _ in hyps]
raw = [math.exp(lp) for _, lp in hyps]
norm = [math.exp(lp / len(h)) for h, lp in hyps]       # bitta soʻzga oʻrtacha ehtimol

fig, ax = plt.subplots(1, 2, figsize=(11, 0.35 * len(hyps) + 1.2), sharey=True)
for a, vals, title in [(ax[0], raw, "P(gipoteza)"), (ax[1], norm, "exp(log P / L): soʻz boshiga")]:
    colors = ["#D9480F" if v == max(vals) else "#2B55C9" for v in vals]
    a.barh(range(len(vals)), vals, color=colors)
    a.set_title(title)
ax[0].set_yticks(range(len(labels)))
ax[0].set_yticklabels(labels)
plt.tight_layout()
plt.show()

Tarjima sifati odatda BLEU metrikasi bilan oʻlchanadi — namunaviy tarjima bilan mos keladigan n-grammlar ulushi. Uni 18-amaliy mashgʻulotda batafsil koʻramiz; bu yerda `nltk` orqali qisqa koʻrinish (qisqa gaplar uchun silliqlash funksiyasi kerak).

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

reference = "я читаю новую книгу".split()
smooth = SmoothingFunction().method1
for k, alpha in [(1, 0.0), (2, 0.0), (4, 1.0)]:
    (h, _), _ = beam_search(lm, k, alpha)
    hyp = list(h[:-1])
    print(f"k={k}, alpha={alpha:g}: {' '.join(hyp):<28} BLEU = {sentence_bleu([reference], hyp, smoothing_function=smooth):.3f}")

## 5. Haqiqiy modelda

`facebook/nllb-200-distilled-600M` — Meta ning ochiq koʻp tilli tarjima modeli (600 mln parametr, fayl ≈2.5 GB, litsenziya CC BY-NC 4.0 — faqat notijorat foydalanish). Unda oʻzbek tili (lotin) `uzn_Latn`, rus tili `rus_Cyrl` kodi bilan belgilangan. Maqsad til kodi `forced_bos_token_id` orqali, nur kengligi `num_beams` orqali, uzunlik boʻyicha meʼyorlash esa `length_penalty` parametri orqali beriladi (standart qiymati 1.0, yaʼni xuddi shu $\log P / L$). Katak faqat Colab da ishlaydi (internet kerak, yuklash bir necha daqiqa davom etadi). Ochkoʻz tarjima nur bilan tarjimadan farq qiladimi va nurda qanday gipotezalar boʻlganini koʻring. Qiziqarli tajriba: `ʻ` belgisini oddiy apostrof `'` ga almashtirsangiz, tarjima oʻzgaradimi?

In [ ]:
# haqiqiy tarjimon uz -> ru; model bir marta yuklab olinadi (≈2.5 GB)
try:
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

    name = "facebook/nllb-200-distilled-600M"
    tok = AutoTokenizer.from_pretrained(name, src_lang="uzn_Latn")
    nllb = AutoModelForSeq2SeqLM.from_pretrained(name)
    tgt_id = tok.convert_tokens_to_ids("rus_Cyrl")

    sentences = ["Men katta uyda yashayman.", "Biz kutubxonada yangi kitob oʻqiyapmiz.",
                 "Samarqand — Buyuk ipak yoʻlidagi qadimiy shahar."]
    for s in sentences:
        inputs = tok(s, return_tensors="pt")
        greedy = nllb.generate(**inputs, forced_bos_token_id=tgt_id, num_beams=1, max_new_tokens=60)
        beam = nllb.generate(**inputs, forced_bos_token_id=tgt_id, num_beams=4, max_new_tokens=60)
        print(s)
        print("   greedy:  ", tok.decode(greedy[0], skip_special_tokens=True))
        print("   beam=4:  ", tok.decode(beam[0], skip_special_tokens=True))

    # birinchi gap uchun nurdagi barcha 4 gipoteza; son = log P / L (length_penalty = 1.0)
    inputs = tok(sentences[0], return_tensors="pt")
    out = nllb.generate(**inputs, forced_bos_token_id=tgt_id, num_beams=4, num_return_sequences=4,
                        output_scores=True, return_dict_in_generate=True, max_new_tokens=60)
    print()
    for seq, sc in zip(out.sequences, out.sequences_scores):
        print(f"   {float(sc):7.3f}  {tok.decode(seq, skip_special_tokens=True)}")
except Exception as e:
    print("Modelni yuklab boʻlmadi (internet va transformers paketi kerak):", e)

## 6. Topshiriqlar

1. `corpus` ga bogʻ haqidagi juftlarni qoʻshing: `("bogʻ", "сад")`, `("katta bogʻ", "большой сад")`, `("bogʻda", "в саду")`, `("bogʻim", "мой сад")`. Ikkala modelni (segmentatsiyasiz va segmentatsiya bilan) qayta oʻqiting. «сад» va «саду» qaysi oʻzbekcha soʻz/morfemalarga eng koʻp bogʻlandi? «bogʻda» iborasi qanday moslashdi?
2. `IBM1` ni 30 iteratsiya oʻqiting va `model.loglik` ning iteratsiyalarga bogʻliqlik grafigini chizing. Qaysi iteratsiyadan keyin haqqoniylik 0.01 dan kam oʻzgaradi?
3. **Simmetrizatsiya.** Modelni teskari yoʻnalishda oʻqiting (korpus juftlaridagi gaplarning oʻrnini almashtiring). «u katta uyda yashaydi» jufti uchun ikkala yoʻnalishdagi bogʻlanishlar toʻplamini (oʻzbekcha soʻz oʻrni, ruscha soʻz oʻrni) oling va ularning kesishmasi va birlashmasini toping. Qaysi bogʻlanishlar faqat birlashmada bor?
4. `beam_search` dagi meʼyorlashni GNMT jarimasi bilan almashtiring: $lp(L) = \left(\frac{5 + L}{6}\right)^{\alpha}$, $\text{score} = \log P / lp(L)$. $\alpha = 0, 0.5, 1, 1.5$ da $k = 4$ li beam search nimani topadi?
5. $k = 1 \dots 5$ da `beam_search` nechta gipotezani (davomni) hisoblashini sanang va toʻliq daraxtdagi tugunlar soni bilan solishtiring. Bu misolda nur toʻliq koʻrib chiqishdan necha marta arzon?

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing